In [89]:
quiet_library <- function(...){suppressPackageStartupMessages(library(...))}
quiet_library(dplyr)
quiet_library(purrr)
quiet_library(tidyr)
quiet_library(data.table)
quiet_library(Seurat)
quiet_library(ggplot2)
quiet_library(glue)
quiet_library(gridExtra)
quiet_library(grid)
quiet_library(ggpubr)

options(repr.matrix.max.cols=150, repr.matrix.max.rows=200, mc.cores = 20, future.globals.maxSize = 2000 * 1024^2)
fig.size <- function (height, width) {
    options(repr.plot.height = height, repr.plot.width = width)
}

In [90]:
wd <- "/home/workspace/IFN"
fig_dir <- file.path(wd, "Figures")

In [91]:
set.seed(123)

In [111]:
stims <- c("IFNa", "IFNb", "IFN-L1", "IFNg")
celltypes <- c("Monocyte", "NK", "Tcell", "Bcell")
subtypes <- c("CD4 Naive", "CD4 Memory", "Treg", "CD8 Naive", "CD8 Memory", "MAIT", "gdT", 
              "B Naive", "B Memory", "Plasma", 
              "NK.CD56hi", "NK.CD56dim", 
              "CD14 Monocyte")
celltype_cols <- c("#00b4d8", "#1b4332", "#ffbf69", "#78290f" )
subtype_cols <- c("#03045e", "#023e8a", "#0077b6", "#0096c7", "#00b4d8", "#48cae4", "#90e0ef",
                   "#1b4332", "#52b788", "#b7e4c7",
                    "#ff9f1c", "#ffbf69", 
                   "#78290f"
                  )
stim_cols <- c("#e9c46a", "#f4a261", "#e76f51", "#264653", "#2a9d8f")
donors <- c("3955BW", "3283BW", "6811BW", "3491BW")

In [93]:
l1_degs <- fread(file.path(wd, "DEGs", "L1_All_Celltypes_Stims_N1_DEGs.csv")) %>%
                filter(Significant == "Yes")
l2_degs <- fread(file.path(wd, "DEGs", "L2_All_Celltypes_Stims_N1_DEGs.csv")) %>%
                filter(Significant == "Yes")

### 2B. Enriched vs PBMC IFNa LogFC Scatterplots

In [96]:
celltype_level <- "L1"

In [153]:
final_df <- fread(file.path(wd, "Batch7", "Enriched_PBMC_IFNa_N1_DEGs_merged.csv"))

In [112]:
# create expression logFC scatter plots per cell type
ggs <- lapply(celltypes, function(c) {
  
  df <- final_df %>% 
        filter(celltype == c) %>% 
        dplyr::filter(!(is.na(Enriched_Log2FC) | is.na(PBMC_Log2FC))) 

  genes_label <- c(
    df %>% slice_max(Enriched_Log2FC, n = 5) %>% pull(gene),
    df %>% slice_max(PBMC_Log2FC, n = 5) %>% pull(gene),
    df %>% slice_min(Enriched_Log2FC, n = 5) %>% pull(gene),
    df %>% slice_min(PBMC_Log2FC, n = 5) %>% pull(gene)
  )

  # additional label on CCL8 in monocytes
  if (c == "Monocyte"){
      genes_label <- c(genes_label, "CCL8")
      }
    
  df <- df %>%
    mutate(
      label = ifelse(gene %in% genes_label, gene, NA),
      color = case_when(
        Enriched_Log2FC > 0 & PBMC_Log2FC > 0 ~ "red",
        Enriched_Log2FC < 0 & PBMC_Log2FC < 0 ~ "blue",
        TRUE ~ "gray"
      )
    )

  
  ggplot(df, aes(x = PBMC_Log2FC, y = Enriched_Log2FC, label = label, color = color)) +
    geom_point(alpha = 1) +
    scale_color_manual(values = c("blue", "gray", "red")) +
    ggrepel::geom_text_repel(max.overlaps = 100) +
    labs(
      x = "PBMC Log2FC",
      y = "Enriched Log2FC",
      title = c
    ) +
    theme_minimal() +
    geom_hline(yintercept = 0, linetype = "solid", linewidth = 0.5) +
      geom_vline(xintercept = 0, linetype = "solid", linewidth = 0.5) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "black", size = 0.5) +
    theme(
      legend.position = "none",
      plot.title = element_text(hjust = 0.5),
        panel.grid = element_blank()
    )
})
 

In [114]:
fig.size(8,9.5)
pdf(file.path(wd, "Figures", "Supp", "Sup2", "PBMC_Enriched_IFNa_Log2FC_Comparison_L1.pdf"), height = 8, width = 9.5)
grid.arrange(grobs = ggs)
dev.off()

Warning message:
“Removed 7598 rows containing missing values or values outside the scale range
(`geom_text_repel()`).”
Warning message:
“Removed 6285 rows containing missing values or values outside the scale range
(`geom_text_repel()`).”
Warning message:
“Removed 5890 rows containing missing values or values outside the scale range
(`geom_text_repel()`).”
Warning message:
“Removed 6876 rows containing missing values or values outside the scale range
(`geom_text_repel()`).”


pdf 
  2

### 2C. Enriched vs PBMC Heatmap

In [154]:
# select genes for heatmap
genes_label <- lapply(celltypes, function(c) {
  
  df <- final_df %>% filter(celltype == c)

  genes_label <- c(
    df %>% slice_max(Enriched_Log2FC, n = 5) %>% pull(gene),
    df %>% slice_max(PBMC_Log2FC, n = 5) %>% pull(gene),
    df %>% slice_min(Enriched_Log2FC, n = 5) %>% pull(gene),
    df %>% slice_min(PBMC_Log2FC, n = 5) %>% pull(gene)
      
      )
    genes_label %>% unique()
    }) 

In [160]:
df %>% fread(file.path(wd, "Batch7", "Donor_Enriched_PBMC_IFNa_N1_DEGs_merged.csv"))

In [143]:
# create donor-level heatmaps per cell type
ggs <- lapply(seq_along(celltypes), function(c){
       
    df_select <- df %>% filter(celltype == celltypes[c])
  
    meta <- df_select %>% select(type, celltype) %>% unique() #%>% select(Subtype)
    mat <- df_select %>% select(gene, iteration_median_log2FC, type_donor) %>% pivot_wider(
                         names_from = type_donor, values_from = iteration_median_log2FC) %>% 
        tibble::column_to_rownames("gene") %>%
        as.matrix()
    mat[is.na(mat)] <- 0

    meta <- df_select %>% dplyr::select(type, donor) %>% unique() #%>% select(Subtype)
    HA <- ComplexHeatmap::HeatmapAnnotation(df_select = meta[, "type"],
                                        col = list(type = c("sorted" = "#0a9396", 
                                                               "unsorted" = "#ddb892"
                  ))
                      )
    
    mat_select <- mat[genes_label[[c]],]
    col_fun <- circlize::colorRamp2(c(min(mat_select), 0, max(mat_select)), c("blue", "gray95", "red"))

   
    p1 <- ComplexHeatmap::Heatmap(
      mat_select,
      name = "log2FC", 
      top_annotation = HA, 
      show_row_names = TRUE, 
      show_column_names = FALSE, 
      col = col_fun, 
      cluster_rows = TRUE,
      column_split = factor(rep(donors, each = 2), levels = donors), 
      cluster_columns = FALSE, 
      cluster_column_slices = FALSE, 
      column_title_rot = 90, 
      show_heatmap_legend = TRUE
    )
    
    p1 %>% ggplotify::as.grob()
    })


In [145]:
fig.size(9,8)
pdf(file.path(wd, "Figures", "Supp", "Sup2",
              "Sorted_Unsorted_L1_IFNa_LogFC_Donor_Heatmap_Genes_Opposite.pdf"), height = 9, width = 9)
grid.arrange(grobs = ggs, ncol = 2)
dev.off()

pdf 
  2

### 2D. Enriched vs PBMC Log2FC PCA

In [62]:
# read-in data for PCA matrix
pca_mat <- map_dfr(celltypes, function(c){
        
        df <- map_dfr(donors, function(d){

            sorted <- fread(file.path(wd, "Batch7", "celltype_DEGs", "Donor_level", "L1", glue("{c}_{d}_IFNa_Sorted_MAST_degs.csv")))
            unsorted <- fread(file.path(wd, "Batch7", "celltype_DEGs", "Donor_level", "L1", glue("{c}_{d}_IFNa_PBMC_MAST_degs.csv")))
          
            sorted$type <- "sorted"
            unsorted$type <- "unsorted"
            df_comb <- rbind(sorted, unsorted)
            df_comb$celltype <- c
            df_comb$donor <- d
            df_comb$type_donor_celltype <- paste0(df_comb$type, df_comb$donor, "_", df_comb$celltype)
            df_comb
        })
    
    meta <- df %>% select(type, celltype) %>% unique() #%>% select(Subtype)
    mat <- df %>% select(!c(p_val, pct.1, pct.2, p_val_adj, type, donor, celltype)) %>% pivot_wider(
                         names_from = type_donor_celltype, values_from = avg_log2FC) %>% tibble::column_to_rownames("gene") %>%
        as.matrix()
    mat[is.na(mat)] <- 0
    
    mat_unsorted <- mat[, grepl("unsorted", colnames(mat))] %>% t()
    sorted_cols <- colnames(mat) %>% setdiff(colnames(mat[, grepl("unsorted", colnames(mat))])) 
    mat_sorted <- mat[, sorted_cols] %>% t()

    
    #rownames(mat_sorted) <- gsub("sorted", "", rownames(mat_sorted))
    rbind(as.data.frame(mat_sorted), as.data.frame(mat_unsorted))
    
    }) 

In [63]:
# run PCA and process output
pca <- prcomp(pca_mat)

pc_scores <- as.data.frame(pca$x[, 1:2])

pc_scores$Name <- rownames(pc_scores)
pc_scores$Celltype <- pc_scores$Name %>% stringr::str_extract("[^_]+$")
pc_scores$Sample <- pc_scores$Name %>% stringr::str_extract("[^_]*")

pc_scores$Celltype <- factor(pc_scores$Celltype, levels = celltypes)
pc_scores$Group <- pc_scores$Name %>% stringr::str_extract("^[^0-9]*")

In [84]:
fig.size(3,4.4)
pdf(file.path(wd, "Figures", "Supp", "Sup2", 'Sorted_Unsorted_PCA.pdf'), height = 3, width = 4.4)
celltype_cols <- c("#00b4d8", "#1b4332", "#ffbf69", "#78290f")
ggplot(pc_scores, aes(x = PC1, y = PC2, color = Celltype, shape = Group)) +
    
    geom_point(size = 2.5) +
    #ggforce::geom_mark_ellipse(aes(fill = Celltype,
    #                    color = Celltype)) + 
    labs(x = "PC1", y = "PC2") +
    theme_classic() + 
    #lims(x = c(-20,45), y = c(-35,30)) + 
    scale_color_manual(values=celltype_cols) + 
    scale_fill_manual(values=celltype_cols) + 
    theme(legend.title = element_blank()) 
dev.off()

pdf 
  2

### 2E. Distance from centroid as variability statistic

In [68]:
# extract PCs
pcax <- as.data.frame(pca$x)

# define groups and celltype and source (enriched, PBMC) combinations
pcax$group <- rownames(pcax) %>% stringr::str_extract("[^_]+$")
pcax$group <- rownames(pcax) %>% gsub("[0-9]{4}BW", "", .)

In [70]:
# get centroid of each cluster
pca_centroids <- pcax %>%
  group_by(group) %>%
  summarise(across(starts_with("PC"), mean), .groups = "drop")


In [71]:
distances <- numeric(nrow(pcax))

# calculate distance of each donor to centroid
for (i in 1:nrow(pcax)) {
  # get the group for this point
  grp <- pcax$group[i]
  point <- pcax[i, c("PC1", "PC2")]
  centroid <- pca_centroids[pca_centroids$group == grp, c("PC1", "PC2")]
  mat <- rbind(point, centroid)

  distances[i] <- as.matrix(dist(mat))[1, 2]
}

# add distance to dataframe
pcax$distance_to_centroid <- distances

# format
pcax$condition <- pcax$group %>% stringr::str_extract("[^_]*")
pcax$celltype <- pcax$group %>% stringr::str_extract("[^_]+$")

In [80]:
fig.size(3,5)
pdf(file.path(wd, "Figures", "Supp", "Sup2", "Sorted_Unsorted_Centroid_Dist_Barplot.pdf"), height = 3, width = 5)
ggplot(pcax, aes(x=condition, y=distance_to_centroid, fill = condition)) +
    geom_boxplot(outlier.shape = NA) +
    geom_jitter(position = position_jitterdodge(jitter.width = 0.2)) +
    theme_classic() + 
    labs(x = "", y = "Average Dist to Centroid") +
    scale_fill_manual(values = c("#0a9396", "#ddb892")) +
    facet_wrap(~celltype, nrow = 1) + 
    ylim(0,40) + 
    stat_compare_means(
      method = "wilcox.test",
      label = "p.signif",       # ← use stars instead of p-value
      size = 4,
      #label.y = 0.7,
      paired = F, 
      comparisons = list(c("sorted", "unsorted"))
    ) + 
    theme(axis.text.x = element_text(angle = 45, vjust = 1, hjust = 1),
          legend.position = "none",
          strip.background = element_blank(),
          panel.spacing = unit(1.2, "lines"))
    
dev.off()

pdf 
  2